In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
apps_cols = pd.read_csv(RAW / "applications.csv", nrows=5).columns
bureau_cols = pd.read_csv(RAW / "credit_bureau.csv", nrows=5).columns
bureau_ids = pd.read_csv(RAW / "credit_bureau.csv", usecols=["id"], dtype={"id": str})["id"]
hold = pd.read_csv(RAW / "holdout_applications.csv", dtype={"id": str}, low_memory=False)

print("holdout shape:", hold.shape)
print("in holdout but not in applications:", sorted(set(hold.columns) - set(apps_cols)))
print("in applications but not in holdout:", sorted(set(apps_cols) - set(hold.columns)))
print("bureau columns present in holdout:", sorted((set(bureau_cols) & set(hold.columns)) - {"id"}))
print("holdout ids found in credit_bureau.csv:", int(hold["id"].isin(bureau_ids).sum()), "of", len(hold))

holdout shape: (5000, 40)
in holdout but not in applications: ['acc_open_past_24mths', 'avg_cur_bal', 'bc_util', 'bureau_recovery_flag', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_high', 'fico_range_low', 'inq_last_6mths', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_rev_hi_lim']
in applications but not in holdout: ['defaulted']
bureau columns present in holdout: ['acc_open_past_24mths', 'avg_cur_bal', 'bc_util', 'bureau_recovery_flag', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_high', 'fico_range_low', 'inq_last_6mths', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_rev_hi_lim']
holdout ids found in credit_bureau.csv: 0 of 5000


In [2]:
import sys
sys.path.append("..")                                   # so `src` can be imported from notebooks/
from src.cleaning import load_training_raw, clean

cleaned = clean(load_training_raw(RAW)).set_index("id")
ref = pd.read_parquet("../data/interim/train_audited.parquet").set_index("id")

print("rows:", len(cleaned), "vs", len(ref))
print("only in module output:", sorted(set(cleaned.columns) - set(ref.columns)))
print("only in notebook output:", sorted(set(ref.columns) - set(cleaned.columns)))

common = [c for c in ref.columns if c in cleaned.columns]
pd.testing.assert_frame_equal(cleaned.loc[ref.index, common], ref[common], check_dtype=False)
print("identical on", len(common), "columns ✅")

flags = ["loan_amnt_corrected", "annual_inc_corrected", "revol_util_corrected", "bc_util_corrected",
         "fico_swapped", "earliest_cr_invalid", "installment_mismatch", "zip_state_mismatch"]
print(cleaned[flags].sum().to_dict())

hold_clean = clean(hold)
print("holdout:", hold_clean.shape, "| leakage column gone:", "bureau_recovery_flag" not in hold_clean.columns)
print("holdout flags:", hold_clean[flags].sum().to_dict())

rows: 30000 vs 30000
only in module output: []
only in notebook output: ['bureau_recovery_flag']
identical on 52 columns ✅
{'loan_amnt_corrected': 300, 'annual_inc_corrected': 899, 'revol_util_corrected': 295, 'bc_util_corrected': 290, 'fico_swapped': 150, 'earliest_cr_invalid': 300, 'installment_mismatch': 11, 'zip_state_mismatch': 27}
holdout: (5000, 52) | leakage column gone: True
holdout flags: {'loan_amnt_corrected': 0, 'annual_inc_corrected': 0, 'revol_util_corrected': 0, 'bc_util_corrected': 0, 'fico_swapped': 0, 'earliest_cr_invalid': 0, 'installment_mismatch': 0, 'zip_state_mismatch': 3}
